# Topic 12 — Unsupervised Machine Learning (Overview)

> **Goal of this notebook:** orient ourselves before the unsupervised topics. Understand what unsupervised learning is, its main families and challenges, and run a quick combined demo (dimensionality reduction + clustering) on real data.

**Contents**
1. What is unsupervised learning?
2. The main families
3. Why it's hard: evaluation without labels
4. A quick tour dataset: handwritten digits
5. Demo: reduce to 2D, then cluster
6. How good are the clusters? (using hidden labels to check)
7. The unsupervised toolbox in this course
8. Summary

## 1. What is Unsupervised Learning?

In **supervised** learning (Topics 1–11) every example came with a label `y`, and we learned a mapping `X → y`. In **unsupervised** learning there are **no labels** — we only have the inputs `X`, and the goal is to discover structure *within the data itself*:

- groups of similar items,
- a lower-dimensional description,
- unusual / rare points,
- relationships between variables.

Because there's no target to predict, there's no single "accuracy" — success is about finding structure that is *useful* or *interpretable*.

## 2. The Main Families

| Family | Question it answers | Example methods |
|--------|---------------------|-----------------|
| **Clustering** | Which items naturally group together? | K-Means, Hierarchical, DBSCAN |
| **Dimensionality reduction** | Can I describe the data with fewer features? | PCA, t-SNE, UMAP |
| **Anomaly detection** | Which points are unusual? | Isolation Forest, One-Class SVM |
| **Association rules** | Which variables co-occur? | Apriori, FP-Growth |

These aren't mutually exclusive — a common pipeline is *reduce dimensions → cluster → inspect outliers*.

## 3. Why It's Hard: Evaluation Without Labels

With no ground truth, evaluation is the central challenge:
- **Internal metrics** judge structure from the data alone — e.g. the **silhouette score** (how tight and well-separated clusters are), inertia, Davies–Bouldin.
- **External metrics** require labels we usually don't have — e.g. **Adjusted Rand Index (ARI)**, used here only because our demo dataset happens to come with labels for checking.
- **Scaling matters** even more than in supervised learning, since most methods are distance- or variance-based.
- The "right" number of clusters / components is itself something you must choose.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 5)
print('Libraries loaded.')

## 4. A Quick Tour Dataset: Handwritten Digits

1,797 images of digits 0–9, each an 8×8 grid = 64 features. We'll **ignore the labels** while learning (that's the unsupervised part) and only peek at them at the end to check our clusters.

In [ ]:
digits = load_digits()
X, y_true = digits.data, digits.target
print('Shape:', X.shape, '(64 pixel features)')

fig, axes = plt.subplots(1, 8, figsize=(10, 2))
for ax, img, lbl in zip(axes, digits.images, digits.target):
    ax.imshow(img, cmap='gray_r'); ax.set_title(str(lbl)); ax.axis('off')
plt.suptitle('Sample digit images'); plt.show()

## 5. Demo: Reduce to 2D, Then Cluster

A classic unsupervised pipeline: standardise → **PCA** to 2 dimensions (for visualisation) → **K-Means** into 10 groups.

In [ ]:
X_s = StandardScaler().fit_transform(X)
X_2d = PCA(n_components=2, random_state=42).fit_transform(X_s)

km = KMeans(n_clusters=10, n_init=10, random_state=42)
clusters = km.fit_predict(X_s)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(X_2d[:, 0], X_2d[:, 1], c=clusters, cmap='tab10', s=8)
axes[0].set_title('Coloured by K-Means cluster (no labels used)')
axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=y_true, cmap='tab10', s=8)
axes[1].set_title('Coloured by true digit (for comparison)')
for a in axes: a.set_xlabel('PC1'); a.set_ylabel('PC2')
plt.show()

## 6. How Good Are the Clusters?

The two plots look similar — K-Means recovered most of the digit structure *without ever seeing a label*. We quantify the agreement with the **Adjusted Rand Index** (1.0 = perfect, 0 = random).

In [ ]:
ari = adjusted_rand_score(y_true, clusters)
print('Adjusted Rand Index (clusters vs true digits):', round(ari, 3))
print('-> well above 0, so the discovered groups align with real digits.')

## 7. The Unsupervised Toolbox in This Course

The remaining topics drill into each tool:

- **PCA** — compress features while keeping variance.
- **K-Means** — partition data into k spherical clusters.
- **Hierarchical clustering** — build a tree (dendrogram) of nested clusters.
- **DBSCAN** — density-based clustering that finds arbitrary shapes and noise.
- **Silhouette analysis** — measure and choose the number of clusters.
- **Anomaly detection** — flag the rare, unusual points.

Each uses the same workflow we just saw: scale → fit → inspect → evaluate.

## 8. Summary

- Unsupervised learning finds structure in **unlabelled** data: clusters, low-dimensional representations, anomalies, and associations.
- Without a target, **evaluation relies on internal metrics** (silhouette, inertia) — external metrics like ARI need labels we usually lack.
- A standard pipeline is **scale → reduce dimensions → cluster → inspect**.
- On handwritten digits, PCA + K-Means recovered the digit groups unsupervised, with a clearly non-random Adjusted Rand Index.